# Fairness Evaluation Notebook
Thesis: How do fairness and ethics mitigation techniques for LLMs affect model performance, and what are the resulting sustainability trade-offs in terms of energy use and computational cost?

This notebook reads all prediction CSV files produced by the baseline and mitigation notebooks and computes:
1. Fairness metrics - Jigsaw Bias AUC (subgroup, BPSN, BNSP), Equality of Opportunity difference per identity group
2. Performance metrics - accuracy, precision, recall, binary F1, weighted F1, AUC-ROC
3. Sustainability metrics - runtime, energy (kWh), CO2 (kg), parameters
4. Trade-off analysis - fairness gain vs. performance cost vs. sustainability cost

Run this notebook after every model/method run to update the results table.

### Expected input files (produced by training notebooks)
Each training notebook saves three CSVs to `baseline_results/`:
```
<model>_<method>_predictions.csv     <- per-sample predictions + identity columns
<model>_<method>_results.csv         <- aggregate performance metrics
<model>_<method>_sustainability.csv  <- energy, CO2, runtime, parameters
```

## 1. Install packages

In [ ]:
%pip install pandas numpy scikit-learn matplotlib seaborn

## 2. Imports

In [ ]:
import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

# Consistent plot style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

## 3. Configuration

Set `RESULTS_DIR` to the same folder used in your training notebooks.  
Add each model/method combination you have run to `EXPECTED_RUNS` - the notebook will warn you if any are missing.

In [ ]:
# Paths
BASE_DIR    = "/Users/medsa/Desktop/finaline/"
RESULTS_DIR = os.path.join(BASE_DIR, "baseline_results")
PLOTS_DIR   = os.path.join(RESULTS_DIR, "final_graphs")
os.makedirs(PLOTS_DIR, exist_ok=True)

# Identity groups to evaluate
IDENTITY_COLUMNS = [
    "male",
    "female",
    "homosexual_gay_or_lesbian",
    "christian",
    "jewish",
    "muslim",
    "black",
    "white",
    "psychiatric_or_mental_illness",
]

IDENTITY_THRESHOLD = 0.5

# All completed runs
EXPECTED_RUNS = [
    # BERT
    ("bert_base_uncased",       "baseline"),
    ("bert_base_uncased",       "prompt_tuning"),
    ("bert_base_uncased",       "adapter"),
    ("bert_base_uncased",       "full_debiasing"),
    # DistilBERT
    ("distilbert_base_uncased", "baseline"),
    ("distilbert_base_uncased", "prompt_tuning"),
    ("distilbert_base_uncased", "adapter"),
    ("distilbert_base_uncased", "full_debiasing"),
    # RoBERTa
    ("roberta_base",            "baseline"),
    ("roberta_base",            "prompt_tuning"),
    ("roberta_base",            "adapter"),
    ("roberta_base",            "full_debiasing"),
]

print(f"Results dir  : {RESULTS_DIR}")
print(f"Plots dir    : {PLOTS_DIR}")
print(f"Expected runs: {len(EXPECTED_RUNS)}")

MIN_GROUP_SIZE = 100   # minimum group size for fairness metrics


## 4. Load all prediction CSV files

Each file is expected to have columns:
- `comment_text`, `true_label`, `toxicity_score`, `prediction`
- Identity columns: `male`, `female`, ..., `psychiatric_or_mental_illness`
- `model`, `method`

In [ ]:
def load_predictions(results_dir, expected_runs):
    """Load all prediction CSVs and concatenate into one DataFrame."""
    dfs = []
    missing = []

    for model_name, method in expected_runs:
        fname = f"{model_name}_{method}_predictions.csv"
        fpath = os.path.join(results_dir, fname)

        if os.path.exists(fpath):
            df = pd.read_csv(fpath)
            df["model_safe"] = model_name
            df["method"]     = method
            dfs.append(df)
            print(f"  Loaded {fname}  ({len(df):,} rows)")
        else:
            missing.append(fpath)
            print(f"  Missing {fname}")

    if missing:
        print(f"\n{len(missing)} file(s) not found - run the corresponding training notebooks first.")

    if dfs:
        combined = pd.concat(dfs, ignore_index=True)
        print(f"\nTotal rows loaded: {len(combined):,}")
        print(f"Runs loaded: {combined.groupby(['model_safe','method']).size().to_dict()}")
        return combined
    else:
        raise FileNotFoundError("No prediction files found. Run training notebooks first.")

all_preds = load_predictions(RESULTS_DIR, EXPECTED_RUNS)

# Per-run lookup used by the diagnostics further down (mutual information, fairness gap table)
all_dfs = {key: group for key, group in all_preds.groupby(["model_safe", "method"])}

all_preds.head(3)

## 5. Verify columns and identity coverage

In [ ]:
# Check required columns exist
required = ["comment_text", "true_label", "toxicity_score", "prediction", "model_safe", "method"]
missing_cols = [c for c in required if c not in all_preds.columns]
if missing_cols:
    print("Missing required columns:", missing_cols)
else:
    print("All required columns present.")

# Check which identity columns are available
available_id_cols = [c for c in IDENTITY_COLUMNS if c in all_preds.columns]
missing_id_cols   = [c for c in IDENTITY_COLUMNS if c not in all_preds.columns]

print(f"\nIdentity columns available : {available_id_cols}")
if missing_id_cols:
    print(f"Identity columns missing   : {missing_id_cols}")

# Class balance per run
print("\nClass balance per run (toxic %):")
print(
    all_preds.groupby(["model_safe", "method"])["true_label"]
    .apply(lambda x: f"{x.mean()*100:.1f}%")
    .reset_index()
    .to_string(index=False)
)

## 6. Jigsaw Bias AUC metric functions

The three Jigsaw Bias AUC metrics measure how well a model ranks toxic vs. non-toxic comments specifically in the context of identity mentions. All three use standard AUC-ROC but on carefully chosen subsets:

| Metric | Subset used | What it detects |
|---|---|---|
| Subgroup AUC | Only comments mentioning the identity group | Whether the model ranks within-group comments correctly |
| BPSN AUC | Non-toxic mentioning group + toxic NOT mentioning group | Whether the model falsely flags identity mentions as toxic |
| BNSP AUC | Toxic mentioning group + non-toxic NOT mentioning group | Whether the model misses toxicity when identity is present |

A score of 1.0 is perfect. Scores below 0.5 mean the model is actively worse than random for that group.

The overall Jigsaw Bias AUC is the power-mean average across all three metrics and all groups, matching the original Kaggle competition scoring formula.

In [ ]:
def safe_auc(y_true, y_score):
    """Return AUC-ROC, or NaN if only one class is present."""
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)


def compute_subgroup_auc(df, identity_col, label_col, score_col, threshold=0.5):
    """AUC on comments that mention the identity group (score >= threshold)."""
    subgroup = df[df[identity_col] >= threshold]
    if len(subgroup) < 10:
        return np.nan
    return safe_auc(subgroup[label_col], subgroup[score_col])


def compute_bpsn_auc(df, identity_col, label_col, score_col, threshold=0.5):
    """
    BPSN = Background Positive Subgroup Negative.
    Subset: non-toxic comments mentioning the group  +  toxic comments NOT mentioning it.
    Low score = model over-flags identity mentions as toxic.
    """
    in_group  = df[identity_col] >= threshold
    bpsn      = df[(in_group & (df[label_col] == 0)) | (~in_group & (df[label_col] == 1))]
    if len(bpsn) < 10:
        return np.nan
    return safe_auc(bpsn[label_col], bpsn[score_col])


def compute_bnsp_auc(df, identity_col, label_col, score_col, threshold=0.5):
    """
    BNSP = Background Negative Subgroup Positive.
    Subset: toxic comments mentioning the group  +  non-toxic comments NOT mentioning it.
    Low score = model fails to detect toxicity when identity is present.
    """
    in_group  = df[identity_col] >= threshold
    bnsp      = df[(in_group & (df[label_col] == 1)) | (~in_group & (df[label_col] == 0))]
    if len(bnsp) < 10:
        return np.nan
    return safe_auc(bnsp[label_col], bnsp[score_col])


def power_mean(values, p=-5):
    """
    Generalised power mean with p=-5, as used by Jigsaw competition.
    Penalises low scores more heavily than a simple average.
    NaN values are ignored.
    """
    valid = [v for v in values if not np.isnan(v)]
    if not valid:
        return np.nan
    return np.power(np.mean(np.power(valid, p)), 1.0 / p)



## 7. Equality of Opportunity difference

Equality of Opportunity checks whether the True Positive Rate (recall for the toxic class) is equal across identity groups. A positive difference means the model catches toxicity *less often* in comments mentioning that group compared to comments that don't - a form of under-detection bias.

In [ ]:
def compute_eod(df, identity_col, label_col, pred_col, threshold=0.5):
    """
    Equality of Opportunity Difference (EOD).
    = TPR(group mentions) - TPR(group not mentioned)

    Positive value: model catches toxic comments mentioning this group MORE often
                    than toxic comments not mentioning it.
    Negative value: model catches toxic comments mentioning this group LESS often.
    Values closer to 0 indicate better equality of opportunity.
    Report both signed (for direction) and absolute (for magnitude) in results.
    """
    in_group  = df[identity_col] >= threshold
    toxic     = df[label_col] == 1

    group_toxic = df[in_group & toxic]
    bg_toxic    = df[~in_group & toxic]

    if len(group_toxic) < 5 or len(bg_toxic) < 5:
        return np.nan

    tpr_group = (group_toxic[pred_col] == 1).mean()
    tpr_bg    = (bg_toxic[pred_col] == 1).mean()

    return round(tpr_group - tpr_bg, 4)


print("  Positive EOD = group has HIGHER recall than background")
print("  Negative EOD = group has LOWER recall than background")

## 8. False Positive Rate parity difference

FPR parity checks whether non-toxic comments mentioning an identity group are falsely labelled as toxic more often than non-toxic comments that don't mention the group. This is the *over-flagging* bias - a key problem in toxicity classifiers.

In [ ]:
def compute_fpr_diff(df, identity_col, label_col, pred_col, threshold=0.5):
    """
    FPR difference.
    = FPR(group mentions) - FPR(group not mentioned)
    Positive value: model over-flags non-toxic identity-mentioning comments as toxic.
    """
    in_group   = df[identity_col] >= threshold
    non_toxic  = df[label_col] == 0

    group_neg = df[in_group & non_toxic]
    bg_neg    = df[~in_group & non_toxic]

    if len(group_neg) < 5 or len(bg_neg) < 5:
        return np.nan

    fpr_group = (group_neg[pred_col] == 1).mean()
    fpr_bg    = (bg_neg[pred_col] == 1).mean()

    return round(fpr_group - fpr_bg, 4)



## 9. Compute all fairness metrics for every run × every identity group

This is the main computation cell. For each (model, method) combination and each identity group it computes:
- Subgroup AUC, BPSN AUC, BNSP AUC
- Equality of Opportunity difference
- FPR parity difference
- Group size (number of comments mentioning the group)

In [ ]:
def compute_fairness_for_run(df, available_id_cols, id_threshold=0.5):
    """
    Compute all per-group fairness metrics for one (model, method) run.
    Returns a list of dicts, one per identity group.
    """
    rows = []
    for col in available_id_cols:
        subgroup_size = int((df[col] >= id_threshold).sum())

        rows.append({
            "identity_group"  : col,
            "subgroup_size"   : subgroup_size,
            "subgroup_auc"    : compute_subgroup_auc(df, col, "true_label", "toxicity_score", id_threshold),
            "bpsn_auc"        : compute_bpsn_auc(df, col, "true_label", "toxicity_score", id_threshold),
            "bnsp_auc"        : compute_bnsp_auc(df, col, "true_label", "toxicity_score", id_threshold),
            "eod"             : compute_eod(df, col, "true_label", "prediction", id_threshold),
            "fpr_diff"        : compute_fpr_diff(df, col, "true_label", "prediction", id_threshold),
        })
    return rows


# Run across all loaded predictions
fairness_rows = []

for (model_safe, method), group_df in all_preds.groupby(["model_safe", "method"]):
    print(f"Computing fairness: {model_safe} / {method}  ({len(group_df):,} rows)")
    per_group = compute_fairness_for_run(group_df, available_id_cols, IDENTITY_THRESHOLD)

    # Proper Jigsaw-style Bias AUC
    # Step 1: separate power mean for each AUC type across all identity groups
    subgroup_vals = [r["subgroup_auc"] for r in per_group if not np.isnan(r["subgroup_auc"] or np.nan)]
    bpsn_vals     = [r["bpsn_auc"]     for r in per_group if not np.isnan(r["bpsn_auc"]     or np.nan)]
    bnsp_vals     = [r["bnsp_auc"]     for r in per_group if not np.isnan(r["bnsp_auc"]     or np.nan)]

    subgroup_pm = power_mean(subgroup_vals) if subgroup_vals else np.nan
    bpsn_pm     = power_mean(bpsn_vals)     if bpsn_vals     else np.nan
    bnsp_pm     = power_mean(bnsp_vals)     if bnsp_vals     else np.nan

    # Step 2: overall Bias Score = mean of the three power means
    bias_components = [v for v in [subgroup_pm, bpsn_pm, bnsp_pm] if not np.isnan(v)]
    overall_bias_auc = np.nanmean(bias_components) if bias_components else np.nan

    for r in per_group:
        r["model_safe"]        = model_safe
        r["method"]            = method
        r["subgroup_power_mean"]= round(subgroup_pm, 4) if not np.isnan(subgroup_pm) else np.nan
        r["bpsn_power_mean"]   = round(bpsn_pm, 4)     if not np.isnan(bpsn_pm)     else np.nan
        r["bnsp_power_mean"]   = round(bnsp_pm, 4)     if not np.isnan(bnsp_pm)     else np.nan
        r["overall_bias_auc"]  = round(overall_bias_auc, 4) if not np.isnan(overall_bias_auc) else np.nan
        fairness_rows.append(r)

fairness_df = pd.DataFrame(fairness_rows)

print(f"\nFairness table: {fairness_df.shape}")
print(fairness_df[["model_safe","method","identity_group",
                   "subgroup_auc","bpsn_auc","bnsp_auc","eod","fpr_diff"]].to_string(index=False))

## 10. Compute overall performance metrics

These are computed directly from the prediction files - so they are consistent regardless of what the training notebook reported internally.

In [ ]:
def compute_performance(df):
    y_true   = df["true_label"].values
    y_pred   = df["prediction"].values
    y_score  = df["toxicity_score"].values

    acc = accuracy_score(y_true, y_pred)
    p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0)
    _, _, f1_w, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0)

    try:
        auc = roc_auc_score(y_true, y_score)
    except ValueError:
        auc = np.nan

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "accuracy"    : round(acc,   4),
        "precision"   : round(p_bin, 4),
        "recall"      : round(r_bin, 4),
        "f1_binary"   : round(f1_bin,4),
        "f1_weighted" : round(f1_w,  4),
        "auc_roc"     : round(auc,   4),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
    }


perf_rows = []
for (model_safe, method), gdf in all_preds.groupby(["model_safe", "method"]):
    row = compute_performance(gdf)
    row["model_safe"] = model_safe
    row["method"]     = method
    perf_rows.append(row)

perf_df = pd.DataFrame(perf_rows)
print("Performance metrics:")
print(perf_df[["model_safe","method","accuracy","precision","recall","f1_binary","f1_weighted","auc_roc"]].to_string(index=False))

## 11. Load sustainability metrics

In [ ]:
sustain_dfs = []

for model_safe, method in EXPECTED_RUNS:
    fpath = os.path.join(RESULTS_DIR, f"{model_safe}_{method}_sustainability.csv")
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        df["model_safe"] = model_safe
        df["method"]     = method
        sustain_dfs.append(df)
        print(f"  {model_safe} / {method}")
    else:
        print(f"  Missing sustainability file for {model_safe} / {method}")

if sustain_dfs:
    sustain_df = pd.concat(sustain_dfs, ignore_index=True)
    display_cols = ["model_safe","method","runtime_seconds","energy_kwh",
                    "emissions_kg_co2eq","trainable_parameters","peak_gpu_memory_gb"]
    available_display = [c for c in display_cols if c in sustain_df.columns]
    print("\nSustainability metrics:")
    print(sustain_df[available_display].to_string(index=False))
else:
    sustain_df = pd.DataFrame()
    print("No sustainability files found yet.")

## 12. Master results table

One row per (model, method). Combines:
- Overall Jigsaw Bias AUC
- Worst-group subgroup AUC (most vulnerable group)
- Mean EOD and FPR diff across groups
- Performance (F1, AUC-ROC)
- Sustainability (runtime, energy, CO2)

This is the core comparison table for your thesis results chapter.

In [ ]:
# Aggregate fairness to one row per (model, method)
fairness_agg = (
    fairness_df
    .groupby(["model_safe", "method"])
    .agg(
        overall_bias_auc   = ("overall_bias_auc", "first"),
        mean_subgroup_auc  = ("subgroup_auc",     "mean"),
        worst_subgroup_auc = ("subgroup_auc",     "min"),
        mean_bpsn_auc      = ("bpsn_auc",         "mean"),
        mean_bnsp_auc      = ("bnsp_auc",         "mean"),
        mean_eod           = ("eod",              "mean"),
        max_abs_eod        = ("eod",              lambda x: x.abs().max()),
        mean_fpr_diff      = ("fpr_diff",         "mean"),
        max_abs_fpr_diff   = ("fpr_diff",         lambda x: x.abs().max()),
    )
    .reset_index()
)

# Merge performance
master = fairness_agg.merge(
    perf_df[["model_safe","method","accuracy","f1_binary","f1_weighted","auc_roc","precision","recall"]],
    on=["model_safe","method"], how="left"
)

# Merge sustainability
if not sustain_df.empty:
    sustain_cols = ["model_safe","method","runtime_seconds","energy_kwh",
                    "emissions_kg_co2eq","trainable_parameters"]
    avail = [c for c in sustain_cols if c in sustain_df.columns]
    master = master.merge(sustain_df[avail], on=["model_safe","method"], how="left")

# Round numeric columns
num_cols = master.select_dtypes(include="number").columns
master[num_cols] = master[num_cols].round(4)

print("Master results table:")
print(master.to_string(index=False))

## 13. Save all results tables

In [ ]:
master_path   = os.path.join(RESULTS_DIR, "master_results.csv")
fairness_path = os.path.join(RESULTS_DIR, "fairness_per_group.csv")
perf_path     = os.path.join(RESULTS_DIR, "performance_results.csv")

master.to_csv(master_path,   index=False)
fairness_df.to_csv(fairness_path, index=False)
perf_df.to_csv(perf_path,    index=False)

print("Saved:")
print(f"  {master_path}")
print(f"  {fairness_path}")
print(f"  {perf_path}")

## 14. Plot - Per-group Bias AUC heatmap

Each row is an identity group. Each column is a (model, method) run.  
Values are subgroup AUC - closer to 1.0 is better.  
This plot shows at a glance which groups are most affected and whether mitigation helps.

In [ ]:
def plot_bias_heatmap(fairness_df, metric="subgroup_auc", title="Subgroup AUC per Identity Group",
                     save_path=None):
    pivot = fairness_df.pivot_table(
        index="identity_group",
        columns=["model_safe", "method"],
        values=metric,
    )
    pivot.columns = [f"{m}\n{meth}" for m, meth in pivot.columns]

    fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns) * 2.2), 5))
    sns.heatmap(
        pivot, annot=True, fmt=".3f", cmap="viridis",
        vmin=0.5, vmax=1.0, linewidths=0.5,
        ax=ax, cbar_kws={"label": metric}
    )
    ax.set_title(title, fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("")
    ax.set_ylabel("Identity group")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


plot_bias_heatmap(
    fairness_df, metric="subgroup_auc",
    title="Subgroup AUC by Identity Group and Method",
    save_path=os.path.join(PLOTS_DIR, "heatmap_subgroup_auc.png")
)

plot_bias_heatmap(
    fairness_df, metric="bpsn_auc",
    title="BPSN AUC (Over-flagging bias) by Identity Group and Method",
    save_path=os.path.join(PLOTS_DIR, "heatmap_bpsn_auc.png")
)

## 15. Plot - Equality of Opportunity difference per group

Bars show EOD per identity group for each method.  
Closer to 0 = fairer. Positive = model detects toxicity less well when identity is mentioned.

In [ ]:
def plot_eod_bars(fairness_df, save_path=None):
    runs = fairness_df[["model_safe","method"]].drop_duplicates()
    n_runs = len(runs)
    fig, axes = plt.subplots(1, n_runs, figsize=(6 * n_runs, 5), sharey=True)

    if n_runs == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, runs.iterrows()):
        sub = fairness_df[
            (fairness_df.model_safe == row.model_safe) &
            (fairness_df.method     == row.method)
        ].copy()
        colors = ["#D55E00" if v > 0 else "#0072B2" for v in sub.eod]
        ax.barh(sub.identity_group, sub.eod, color=colors)
        ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
        ax.set_title(f"{row.model_safe}\n{row.method}", fontsize=10)
        ax.set_xlabel("EOD (+ = under-detection bias)")

    axes[0].set_ylabel("Identity group")
    fig.suptitle("Equality of Opportunity Difference per Identity Group", fontweight="bold")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


plot_eod_bars(
    fairness_df,
    save_path=os.path.join(PLOTS_DIR, "eod_bars.png")
)

## 16. Plot - Fairness vs. Performance trade-off scatter

Each point is one (model, method) run.  
- X axis: F1 binary (performance - higher is better)  
- Y axis: Overall Jigsaw Bias AUC (fairness - higher is better)  
- Bubble size: CO2 emissions or runtime (sustainability cost)  

The ideal result is the top-right corner - high fairness AND high performance.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D


def plot_tradeoff(master, save_path=None):
    if master.empty:
        print("No data to plot yet.")
        return

    required_cols = ["model_safe", "method", "f1_binary", "overall_bias_auc"]
    missing = [c for c in required_cols if c not in master.columns]

    if missing:
        print(f"Missing columns: {missing}")
        return

    df = master.copy()

    # ---------------------------------------------------------
    # Labels and style
    # ---------------------------------------------------------
    MODEL_COLORS = {
        "bert_base_uncased": "#0072B2",
        "distilbert_base_uncased": "#E69F00",
        "roberta_base": "#009E73",
    }

    MODEL_LABELS = {
        "bert_base_uncased": "BERT",
        "distilbert_base_uncased": "DistilBERT",
        "roberta_base": "RoBERTa",
    }

    METHOD_MARKERS = {
        "baseline": "o",
        "prompt_tuning": "s",
        "adapter": "^",
        "full_debiasing": "D",
    }

    METHOD_LABELS = {
        "baseline": "Baseline",
        "prompt_tuning": "Prompt tuning",
        "adapter": "Adapter",
        "full_debiasing": "Full fine-tuning",
    }

    model_order = [
        "bert_base_uncased",
        "distilbert_base_uncased",
        "roberta_base",
    ]

    method_order = [
        "baseline",
        "prompt_tuning",
        "adapter",
        "full_debiasing",
    ]

    model_order = [m for m in model_order if m in df["model_safe"].unique()]
    method_order = [m for m in method_order if m in df["method"].unique()]

    # ---------------------------------------------------------
    # Bubble size: energy preferred, runtime fallback
    # ---------------------------------------------------------
    if "energy_kwh" in df.columns and df["energy_kwh"].notna().any():
        size_col = "energy_kwh"
        size_label = "Energy (kWh)"
        size_unit = "kWh"
    elif "runtime_seconds" in df.columns and df["runtime_seconds"].notna().any():
        size_col = "runtime_seconds"
        size_label = "Runtime (s)"
        size_unit = "s"
    else:
        size_col = None
        size_label = ""
        size_unit = ""

    if size_col:
        values = df[size_col].fillna(df[size_col].median())
        vmin, vmax = values.min(), values.max()

        if np.isclose(vmin, vmax):
            df["_bubble_size"] = 500
        else:
            df["_bubble_size"] = 250 + 550 * (values - vmin) / (vmax - vmin)
    else:
        df["_bubble_size"] = 450

    # ---------------------------------------------------------
    # Plot
    # ---------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10.5, 6.5))

    for _, row in df.iterrows():
        model = row["model_safe"]
        method = row["method"]

        ax.scatter(
            row["f1_binary"],
            row["overall_bias_auc"],
            s=row["_bubble_size"],
            color=MODEL_COLORS.get(model, "grey"),
            marker=METHOD_MARKERS.get(method, "o"),
            alpha=0.78,
            edgecolors="black",
            linewidths=0.8,
            zorder=3,
        )

    # ---------------------------------------------------------
    # Point labels
    # ---------------------------------------------------------
    label_offsets = {
        "baseline": (7, 5),
        "prompt_tuning": (7, -10),
        "adapter": (7, 7),
        "full_debiasing": (7, -13),
    }

    for _, row in df.iterrows():
        method = row["method"]
        label = METHOD_LABELS.get(method, method.replace("_", " "))
        dx, dy = label_offsets.get(method, (7, 5))

        ax.annotate(
            label,
            (row["f1_binary"], row["overall_bias_auc"]),
            textcoords="offset points",
            xytext=(dx, dy),
            fontsize=8,
            alpha=0.9,
            bbox=dict(
                boxstyle="round,pad=0.15",
                fc="white",
                ec="none",
                alpha=0.65
            ),
            zorder=4,
        )

    # ---------------------------------------------------------
    # Axis padding
    # ---------------------------------------------------------
    x_min, x_max = df["f1_binary"].min(), df["f1_binary"].max()
    y_min, y_max = df["overall_bias_auc"].min(), df["overall_bias_auc"].max()

    x_pad = max(0.005, (x_max - x_min) * 0.12)
    y_pad = max(0.003, (y_max - y_min) * 0.15)

    ax.set_xlim(x_min - x_pad, x_max + x_pad)
    ax.set_ylim(y_min - y_pad, y_max + y_pad)

    # ---------------------------------------------------------
    # Legends outside the plot
    # ---------------------------------------------------------
    model_handles = [
        mpatches.Patch(
            color=MODEL_COLORS[m],
            label=MODEL_LABELS.get(m, m.replace("_", " "))
        )
        for m in model_order
    ]

    method_handles = [
        Line2D(
            [0], [0],
            marker=METHOD_MARKERS[m],
            color="black",
            markerfacecolor="white",
            markeredgecolor="black",
            linestyle="None",
            markersize=8,
            label=METHOD_LABELS.get(m, m.replace("_", " "))
        )
        for m in method_order
    ]

    legend1 = ax.legend(
        handles=model_handles,
        title="Model",
        loc="upper left",
        bbox_to_anchor=(1.02, 1.00),
        fontsize=8,
        title_fontsize=10,
        frameon=True
    )

    ax.add_artist(legend1)

    legend2 = ax.legend(
        handles=method_handles,
        title="Method",
        loc="upper left",
        bbox_to_anchor=(1.02, 0.67),
        fontsize=8,
        title_fontsize=10,
        frameon=True
    )

    ax.add_artist(legend2)

    if size_col:
        ref_vals = sorted({
            round(df[size_col].min(), 3),
            round(df[size_col].median(), 3),
            round(df[size_col].max(), 3),
        })

        ref_sizes = []

        for v in ref_vals:
            if np.isclose(df[size_col].min(), df[size_col].max()):
                s = 500
            else:
                s = 250 + 550 * (v - df[size_col].min()) / (
                    df[size_col].max() - df[size_col].min()
                )

            ref_sizes.append(s)

        size_handles = [
            ax.scatter(
                [],
                [],
                s=s,
                color="grey",
                alpha=0.55,
                edgecolors="black",
                linewidths=0.7,
                label=f"{v:.3f} {size_unit}"
            )
            for s, v in zip(ref_sizes, ref_vals)
        ]

        ax.legend(
            handles=size_handles,
            title=size_label,
            loc="upper left",
            bbox_to_anchor=(1.02, 0.35),
            fontsize=8,
            title_fontsize=10,
            frameon=True
        )

    # ---------------------------------------------------------
    # Labels and title
    # ---------------------------------------------------------
    ax.set_xlabel("Binary F1 score (higher is better)", fontsize=11)
    ax.set_ylabel("Overall Jigsaw Bias AUC (higher is fairer)", fontsize=11)

    ax.set_title(
        "Fairness--Performance Trade-off",
        fontsize=14,
        fontweight="bold",
        pad=12
    )

    ax.grid(True, alpha=0.25)
    ax.set_axisbelow(True)

    plt.tight_layout(rect=[0, 0, 0.78, 1])

    if save_path:
        fig.savefig(save_path, bbox_inches="tight", dpi=300)
        print(f"Saved: {save_path}")

    plt.show()


plot_tradeoff(
    master,
    save_path=os.path.join(PLOTS_DIR, "tradeoff_scatter_clean.png")
)

## 17. Plot - Sustainability comparison

Runtime and CO2 per (model, method). Shows the computational cost of each mitigation technique relative to baseline.

In [ ]:
def plot_sustainability(master, save_path=None):
    if sustain_df.empty or master.empty:
        print("No sustainability data to plot yet.")
        return

    metrics = []
    labels  = []
    if "runtime_seconds" in master.columns:
        metrics.append("runtime_seconds")
        labels.append("Runtime (s)")
    if "energy_kwh" in master.columns and master["energy_kwh"].notna().any():
        metrics.append("energy_kwh")
        labels.append("Energy (kWh)")
    if "emissions_kg_co2eq" in master.columns and master["emissions_kg_co2eq"].notna().any():
        metrics.append("emissions_kg_co2eq")
        labels.append("CO2 (kg)")

    if not metrics:
        print("No numeric sustainability metrics available yet.")
        return

    n = len(metrics)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]

    MODEL_COLORS = {
        "bert_base_uncased"       : "#0072B2",  # blue
        "distilbert_base_uncased" : "#E69F00",  # orange
        "roberta_base"            : "#009E73",  # bluish green
    }
    run_labels = master["model_safe"] + "\n" + master["method"]
    bar_colors = [MODEL_COLORS.get(m, "#999999") for m in master["model_safe"]]

    for ax, metric, label in zip(axes, metrics, labels):
        vals = master[metric].fillna(0)
        ax.bar(run_labels, vals, color=bar_colors, edgecolor="white")
        ax.set_title(label, fontweight="bold")
        ax.set_ylabel(label)
        ax.tick_params(axis="x", rotation=30)

    fig.suptitle("Sustainability Cost per Method", fontweight="bold", y=1.02)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


plot_sustainability(
    master,
    save_path=os.path.join(PLOTS_DIR, "sustainability_bars.png")
)

## 18. Delta analysis - change relative to baseline

Every metric is expressed as delta vs. baseline for the same model.  
This is the clearest way to answer your research question:

> "How much fairness did we gain, at what performance cost, and at what sustainability cost?"

A positive delta on Bias AUC = fairness improved.  
A negative delta on F1 = performance degraded.  
A positive delta on energy = more compute was used.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D


def plot_tradeoff(master, save_path=None):
    if master.empty:
        print("No data to plot yet.")
        return

    required_cols = ["model_safe", "method", "f1_binary", "overall_bias_auc"]
    missing = [c for c in required_cols if c not in master.columns]

    if missing:
        print(f"Missing columns: {missing}")
        return

    df = master.copy()

    # ---------------------------------------------------------
    # Labels and style
    # ---------------------------------------------------------
    MODEL_COLORS = {
        "bert_base_uncased": "#0072B2",
        "distilbert_base_uncased": "#E69F00",
        "roberta_base": "#009E73",
    }

    MODEL_LABELS = {
        "bert_base_uncased": "BERT",
        "distilbert_base_uncased": "DistilBERT",
        "roberta_base": "RoBERTa",
    }

    METHOD_MARKERS = {
        "baseline": "o",
        "prompt_tuning": "s",
        "adapter": "^",
        "full_debiasing": "D",
    }

    METHOD_LABELS = {
        "baseline": "Baseline",
        "prompt_tuning": "Prompt tuning",
        "adapter": "Adapter",
        "full_debiasing": "Full fine-tuning",
    }

    model_order = [
        "bert_base_uncased",
        "distilbert_base_uncased",
        "roberta_base",
    ]

    method_order = [
        "baseline",
        "prompt_tuning",
        "adapter",
        "full_debiasing",
    ]

    model_order = [m for m in model_order if m in df["model_safe"].unique()]
    method_order = [m for m in method_order if m in df["method"].unique()]

    # ---------------------------------------------------------
    # Bubble size: energy preferred, runtime fallback
    # ---------------------------------------------------------
    if "energy_kwh" in df.columns and df["energy_kwh"].notna().any():
        size_col = "energy_kwh"
        size_label = "Energy (kWh)"
        size_unit = "kWh"
    elif "runtime_seconds" in df.columns and df["runtime_seconds"].notna().any():
        size_col = "runtime_seconds"
        size_label = "Runtime (s)"
        size_unit = "s"
    else:
        size_col = None
        size_label = ""
        size_unit = ""

    if size_col:
        values = df[size_col].fillna(df[size_col].median())
        vmin, vmax = values.min(), values.max()

        if np.isclose(vmin, vmax):
            df["_bubble_size"] = 500
        else:
            df["_bubble_size"] = 250 + 550 * (values - vmin) / (vmax - vmin)
    else:
        df["_bubble_size"] = 450

    # ---------------------------------------------------------
    # Plot
    # ---------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10.5, 6.5))

    for _, row in df.iterrows():
        model = row["model_safe"]
        method = row["method"]

        ax.scatter(
            row["f1_binary"],
            row["overall_bias_auc"],
            s=row["_bubble_size"],
            color=MODEL_COLORS.get(model, "grey"),
            marker=METHOD_MARKERS.get(method, "o"),
            alpha=0.78,
            edgecolors="black",
            linewidths=0.8,
            zorder=3,
        )

    # ---------------------------------------------------------
    # Point labels
    # ---------------------------------------------------------
    label_offsets = {
        "baseline": (7, 5),
        "prompt_tuning": (7, -10),
        "adapter": (7, 7),
        "full_debiasing": (7, -13),
    }

    for _, row in df.iterrows():
        method = row["method"]
        label = METHOD_LABELS.get(method, method.replace("_", " "))
        dx, dy = label_offsets.get(method, (7, 5))

        ax.annotate(
            label,
            (row["f1_binary"], row["overall_bias_auc"]),
            textcoords="offset points",
            xytext=(dx, dy),
            fontsize=8,
            alpha=0.9,
            bbox=dict(
                boxstyle="round,pad=0.15",
                fc="white",
                ec="none",
                alpha=0.65
            ),
            zorder=4,
        )

    # ---------------------------------------------------------
    # Axis padding
    # ---------------------------------------------------------
    x_min, x_max = df["f1_binary"].min(), df["f1_binary"].max()
    y_min, y_max = df["overall_bias_auc"].min(), df["overall_bias_auc"].max()

    x_pad = max(0.005, (x_max - x_min) * 0.12)
    y_pad = max(0.003, (y_max - y_min) * 0.15)

    ax.set_xlim(x_min - x_pad, x_max + x_pad)
    ax.set_ylim(y_min - y_pad, y_max + y_pad)

    # ---------------------------------------------------------
    # Legends outside the plot
    # ---------------------------------------------------------
    model_handles = [
        mpatches.Patch(
            color=MODEL_COLORS[m],
            label=MODEL_LABELS.get(m, m.replace("_", " "))
        )
        for m in model_order
    ]

    method_handles = [
        Line2D(
            [0], [0],
            marker=METHOD_MARKERS[m],
            color="black",
            markerfacecolor="white",
            markeredgecolor="black",
            linestyle="None",
            markersize=8,
            label=METHOD_LABELS.get(m, m.replace("_", " "))
        )
        for m in method_order
    ]

    legend1 = ax.legend(
        handles=model_handles,
        title="Model",
        loc="upper left",
        bbox_to_anchor=(1.02, 1.00),
        fontsize=8,
        title_fontsize=10,
        frameon=True
    )

    ax.add_artist(legend1)

    legend2 = ax.legend(
        handles=method_handles,
        title="Method",
        loc="upper left",
        bbox_to_anchor=(1.02, 0.67),
        fontsize=8,
        title_fontsize=10,
        frameon=True
    )

    ax.add_artist(legend2)

    if size_col:
        ref_vals = sorted({
            round(df[size_col].min(), 3),
            round(df[size_col].median(), 3),
            round(df[size_col].max(), 3),
        })

        ref_sizes = []

        for v in ref_vals:
            if np.isclose(df[size_col].min(), df[size_col].max()):
                s = 500
            else:
                s = 250 + 550 * (v - df[size_col].min()) / (
                    df[size_col].max() - df[size_col].min()
                )

            ref_sizes.append(s)

        size_handles = [
            ax.scatter(
                [],
                [],
                s=s,
                color="grey",
                alpha=0.55,
                edgecolors="black",
                linewidths=0.7,
                label=f"{v:.3f} {size_unit}"
            )
            for s, v in zip(ref_sizes, ref_vals)
        ]

        ax.legend(
            handles=size_handles,
            title=size_label,
            loc="upper left",
            bbox_to_anchor=(1.02, 0.35),
            fontsize=8,
            title_fontsize=10,
            frameon=True
        )

    # ---------------------------------------------------------
    # Labels and title
    # ---------------------------------------------------------
    ax.set_xlabel("Binary F1 score ", fontsize=11)
    ax.set_ylabel("Overall Jigsaw Bias AUC ", fontsize=11)

    ax.set_title(
        "Fairness VS Performance Trade-off",
        fontsize=14,
        fontweight="bold",
        pad=12
    )

    ax.grid(True, alpha=0.25)
    ax.set_axisbelow(True)

    plt.tight_layout(rect=[0, 0, 0.78, 1])

    if save_path:
        fig.savefig(save_path, bbox_inches="tight", dpi=300)
        print(f"Saved: {save_path}")

    plt.show()


plot_tradeoff(
    master,
    save_path=os.path.join(PLOTS_DIR, "tradeoff_scatter_clean.png")
)

## 19. Worst-group analysis

Which identity group is most consistently disadvantaged across methods?  
This table shows mean subgroup AUC and mean EOD per identity group, averaged across all runs.

In [ ]:
worst_group = (
    fairness_df.groupby("identity_group")
    .agg(
        mean_subgroup_auc  = ("subgroup_auc", "mean"),
        min_subgroup_auc   = ("subgroup_auc", "min"),
        mean_bpsn_auc      = ("bpsn_auc",     "mean"),
        mean_eod           = ("eod",          "mean"),
        mean_fpr_diff      = ("fpr_diff",     "mean"),
        n_runs             = ("subgroup_auc", "count"),
    )
    .sort_values("mean_subgroup_auc")
    .round(4)
    .reset_index()
)

print("Worst-group analysis (sorted by mean subgroup AUC, lowest first):")
print(worst_group.to_string(index=False))

## 20. Mutual Information - prediction dependence on identity

Mutual information MI(Y_hat, A) measures how much the model's predictions
depend on identity group membership. A value of 0 means the model's predictions
are completely independent of the identity group. Higher values indicate the model
uses identity membership as a signal - quantifying the shortcut-learning risk.

This is computed per identity group per run and reported alongside the Jigsaw Bias
AUC metrics. It complements FPR difference by showing whether the dependency exists
at the probability level rather than just at the binary prediction level.

In [ ]:
def entropy(y):
    """Shannon entropy H(Y) in bits."""
    vals, counts = np.unique(np.asarray(y), return_counts=True)
    probs = counts / counts.sum()
    return -np.sum(probs * np.log2(probs + 1e-12))


def mutual_information(y1, y2):
    """
    Mutual information MI(Y1; Y2) in bits.
    Measures how much knowing Y2 reduces uncertainty about Y1.
    For MI(Y_hat, A): how much does identity group membership tell us
    about the model's predictions?
    """
    y1, y2 = np.asarray(y1), np.asarray(y2)
    vals1, vals2 = np.unique(y1), np.unique(y2)
    n  = len(y1)
    mi = 0.0
    for v1 in vals1:
        for v2 in vals2:
            p_joint = np.sum((y1 == v1) & (y2 == v2)) / n
            p1      = np.sum(y1 == v1) / n
            p2      = np.sum(y2 == v2) / n
            if p_joint > 0:
                mi += p_joint * np.log2(p_joint / (p1 * p2 + 1e-12))
    return max(mi, 0.0)


def compute_mi_for_run(df, identity_cols, id_threshold=0.5, min_size=100):
    """
    Compute MI(prediction, identity_group) for each identity group.
    Returns a dict {group: mi_value}.
    """
    mi_results = {}
    preds = df["prediction"].values

    for col in identity_cols:
        if col not in df.columns:
            continue
        group_membership = (df[col].fillna(0) >= id_threshold).astype(int).values
        if group_membership.sum() < min_size:
            continue
        mi = mutual_information(preds, group_membership)
        mi_results[col] = round(mi, 6)

    return mi_results


# Compute MI for all runs
mi_rows = []
for (model_safe, method), df in all_dfs.items():
    id_cols = [c for c in IDENTITY_COLUMNS if c in df.columns]
    mi_dict = compute_mi_for_run(df, id_cols)
    for group, mi_val in mi_dict.items():
        mi_rows.append({
            "model_safe"    : model_safe,
            "method"        : method,
            "identity_group": group,
            "mutual_information": mi_val,
        })

mi_df = pd.DataFrame(mi_rows)

print("Mutual Information MI(prediction, identity_group):")
print(mi_df.pivot_table(
    index="identity_group",
    columns=["model_safe","method"],
    values="mutual_information"
).round(5).to_string())

mi_path = os.path.join(RESULTS_DIR, "mutual_information.csv")
mi_df.to_csv(mi_path, index=False)
print(f"\nSaved: {mi_path}")

## 21. Fairness gap summary table

Per-identity-group table showing Equality of Opportunity Difference and
FPR Difference for every (model, method) combination. Positive EOD means
the group has higher recall than background. Positive FPRD means the group
has a higher false positive rate - the primary bias signal in this dataset.

In [ ]:
gap_rows = []

for (model_safe, method), df in all_dfs.items():
    id_cols = [c for c in IDENTITY_COLUMNS if c in df.columns]

    for col in id_cols:
        group_mask = (df[col].fillna(0) >= IDENTITY_THRESHOLD)
        if group_mask.sum() < MIN_GROUP_SIZE:
            continue

        # EOD
        eod_val = compute_eod(df, col, "true_label", "prediction")

        # FPRD
        fpr_val = compute_fpr_diff(df, col, "true_label", "prediction")

        gap_rows.append({
            "model"          : model_safe,
            "method"         : method,
            "identity_group" : col,
            "n_examples"     : int(group_mask.sum()),
            "eod"            : round(eod_val, 4) if not np.isnan(eod_val) else None,
            "fprd"           : round(fpr_val, 4) if not np.isnan(fpr_val) else None,
            "abs_eod"        : round(abs(eod_val), 4) if not np.isnan(eod_val) else None,
            "abs_fprd"       : round(abs(fpr_val), 4) if not np.isnan(fpr_val) else None,
        })

gap_df = pd.DataFrame(gap_rows)

# Summary pivot - FPRD per group per method
print("=== FPR Difference per identity group ===")
print("(positive = group over-flagged relative to background)")
print()
fprd_pivot = gap_df.pivot_table(
    index="identity_group",
    columns=["model","method"],
    values="fprd"
).round(4)
print(fprd_pivot.to_string())

gap_path = os.path.join(RESULTS_DIR, "fairness_gap_table.csv")
gap_df.to_csv(gap_path, index=False)
print(f"\nSaved: {gap_path}")

## 22. Disadvantaged group identification

For each (model, method), which identity group has the highest FPR difference
and lowest TPR? This tells you which group is most consistently harmed by
the model's predictions and whether mitigation methods help that specific group.

In [ ]:
disadv_rows = []

for (model_safe, method), group_data in gap_df.groupby(["model","method"]):
    group_data = group_data.dropna(subset=["fprd","eod"])

    if group_data.empty:
        continue

    worst_fprd = group_data.loc[group_data["fprd"].idxmax()]
    worst_eod  = group_data.loc[group_data["eod"].idxmin()]   # most negative = lowest recall

    disadv_rows.append({
        "model"                      : model_safe,
        "method"                     : method,
        "worst_fprd_group"           : worst_fprd["identity_group"],
        "worst_fprd_value"           : worst_fprd["fprd"],
        "lowest_recall_group"        : worst_eod["identity_group"],
        "lowest_recall_eod"          : worst_eod["eod"],
    })

disadv_df = pd.DataFrame(disadv_rows)

print("=== Most disadvantaged group per (model, method) ===")
print()
print(disadv_df.to_string(index=False))

disadv_path = os.path.join(RESULTS_DIR, "disadvantaged_groups.csv")
disadv_df.to_csv(disadv_path, index=False)
print(f"\nSaved: {disadv_path}")

## 23. Accuracy vs Fairness incompatibility plot

Each point is one (model, method) combination. The X-axis shows the
absolute fairness gap (mean absolute FPRD across identity groups).
The Y-axis shows binary F1. Points in the top-left are ideal - high
performance, low bias.

In [ ]:
# Compute mean absolute FPRD per run
mean_fprd = gap_df.groupby(["model","method"])["abs_fprd"].mean().reset_index()
mean_fprd.columns = ["model_safe","method","mean_abs_fprd"]

# Merge with performance
plot_df = master.merge(mean_fprd, on=["model_safe","method"], how="left")
plot_df = plot_df.dropna(subset=["mean_abs_fprd","f1_binary"])

if plot_df.empty:
    print("No data available for plot yet - run training notebooks first.")
else:
    MODEL_COLORS = {
        "bert_base_uncased"       : "#0072B2",  # blue
        "distilbert_base_uncased" : "#E69F00",  # orange
        "roberta_base"            : "#009E73",  # bluish green
    }
    METHOD_MARKERS = {
        "baseline"           : "o",
        "prompt_tuning"      : "s",
        "adapter"            : "^",
        "full_debiasing"     : "D",
    }

    fig, ax = plt.subplots(figsize=(10, 7))

    for _, row in plot_df.iterrows():
        color  = MODEL_COLORS.get(row["model_safe"], "grey")
        marker = METHOD_MARKERS.get(row["method"], "x")
        ax.scatter(row["mean_abs_fprd"], row["f1_binary"],
                   color=color, marker=marker, s=120,
                   edgecolors="black", linewidths=0.6, zorder=3)
        ax.annotate(
            f"{row['method'].replace('full_debiasing','full FT').replace('prompt_tuning','prompt')}\n{row['model_safe'][:4]}",
            (row["mean_abs_fprd"], row["f1_binary"]),
            textcoords="offset points", xytext=(6, 4),
            fontsize=7, color=color
        )

    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0],[0], marker="o", color="w", markerfacecolor=c,
               markersize=10, label=m.replace("_"," "))
        for m, c in MODEL_COLORS.items()
    ] + [
        Line2D([0],[0], marker=mk, color="grey", markersize=9, label=mt.replace("_"," "))
        for mt, mk in METHOD_MARKERS.items()
    ]
    ax.legend(handles=legend_elements, fontsize=8, loc="lower left")

    ax.set_xlabel("Mean Absolute FPR Difference ", fontsize=12)
    ax.set_ylabel("Binary F1 Score", fontsize=12)
    ax.set_title("Performance vs Fairness Trade-off\nAll Models and Mitigation Methods",
                 fontweight="bold", fontsize=13)

    plt.tight_layout()
    save_path = os.path.join(PLOTS_DIR, "tradeoff_accuracy_vs_fairness.png")
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

## 24. F1 by model and method (grouped bar, and line-per-model)

Two views of the same performance numbers already in `master`: a grouped bar chart
(F1 by model, one bar per method) and a line plot (F1 across methods, one line per model).

In [ ]:
def plot_f1_by_model_method(master, save_path_bar=None, save_path_line=None):
    if master.empty:
        print("No data to plot yet.")
        return

    METHOD_ORDER  = ["baseline", "adapter", "prompt_tuning", "full_debiasing"]
    METHOD_LABELS = {
        "baseline": "Baseline", "adapter": "Adapter",
        "prompt_tuning": "Prompt tuning", "full_debiasing": "Full fine-tuning",
    }
    METHOD_COLORS = {
        "baseline"      : "#7F7F7F",  # grey (reference condition)
        "adapter"       : "#56B4E9",  # sky blue
        "prompt_tuning" : "#E69F00",  # orange
        "full_debiasing": "#CC79A7",  # reddish purple
    }

    # --- Figure 5: grouped bar, F1 by model and method ---
    pivot = master.pivot_table(index="model_safe", columns="method", values="f1_binary")
    pivot = pivot.reindex(columns=[m for m in METHOD_ORDER if m in pivot.columns])

    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(pivot.index))
    width = 0.8 / len(pivot.columns)
    for i, method in enumerate(pivot.columns):
        ax.bar(x + i * width, pivot[method], width,
               label=METHOD_LABELS.get(method, method),
               color=METHOD_COLORS.get(method, "#999999"))
    ax.set_xticks(x + width * (len(pivot.columns) - 1) / 2)
    ax.set_xticklabels(pivot.index)
    ax.set_ylabel("F1 (binary)")
    ax.set_title("F1 (binary) by model and method", fontweight="bold")
    ax.legend()
    plt.tight_layout()
    if save_path_bar:
        fig.savefig(save_path_bar, bbox_inches="tight")
        print(f"Saved: {save_path_bar}")
    plt.show()

    # --- Figure 6: line plot, F1 across methods, one line per model ---
    fig, ax = plt.subplots(figsize=(8, 5))
    for model_name in master["model_safe"].unique():
        sub = master[master["model_safe"] == model_name].set_index("method")
        sub = sub.reindex([m for m in METHOD_ORDER if m in sub.index])
        ax.plot(
            [METHOD_LABELS.get(m, m) for m in sub.index],
            sub["f1_binary"],
            marker="o", label=model_name,
        )
    ax.set_ylabel("F1 (binary)")
    ax.set_title("F1 (binary) across methods, by model", fontweight="bold")
    ax.legend(title="Model")
    plt.tight_layout()
    if save_path_line:
        fig.savefig(save_path_line, bbox_inches="tight")
        print(f"Saved: {save_path_line}")
    plt.show()


plot_f1_by_model_method(
    master,
    save_path_bar=os.path.join(PLOTS_DIR, "f1_by_model_method_bar.png"),
    save_path_line=os.path.join(PLOTS_DIR, "f1_by_model_method_line.png"),
)

## 25. Mean absolute FPRD by model and method

Mean of |FPRD| across all nine identity groups (no small-group filtering here, unlike the
seven-group comparison used later for the F1-vs-FPRD trade-off plot). Computed fresh from
`fairness_df` because `master`'s existing `mean_fpr_diff` column is a signed mean, which can
partially cancel across groups the same way mean EOD does.

In [ ]:
def plot_fprd_by_model_method(fairness_df, save_path=None):
    if fairness_df.empty:
        print("No data to plot yet.")
        return

    METHOD_ORDER  = ["baseline", "adapter", "prompt_tuning", "full_debiasing"]
    METHOD_LABELS = {
        "baseline": "Baseline", "adapter": "Adapter",
        "prompt_tuning": "Prompt tuning", "full_debiasing": "Full fine-tuning",
    }
    METHOD_COLORS = {
        "baseline"      : "#7F7F7F",
        "adapter"       : "#56B4E9",
        "prompt_tuning" : "#E69F00",
        "full_debiasing": "#CC79A7",
    }

    mean_abs_fprd = (
        fairness_df
        .groupby(["model_safe", "method"])["fpr_diff"]
        .apply(lambda s: s.abs().mean())
        .reset_index(name="mean_abs_fprd")
    )
    pivot = mean_abs_fprd.pivot_table(index="model_safe", columns="method", values="mean_abs_fprd")
    pivot = pivot.reindex(columns=[m for m in METHOD_ORDER if m in pivot.columns])

    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(pivot.index))
    width = 0.8 / len(pivot.columns)
    for i, method in enumerate(pivot.columns):
        ax.bar(x + i * width, pivot[method], width,
               label=METHOD_LABELS.get(method, method),
               color=METHOD_COLORS.get(method, "#999999"))
    ax.set_xticks(x + width * (len(pivot.columns) - 1) / 2)
    ax.set_xticklabels(pivot.index)
    ax.set_ylabel("Mean absolute FPRD")
    ax.set_title("Mean absolute FPRD by model and method", fontweight="bold")
    ax.legend()
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


plot_fprd_by_model_method(
    fairness_df,
    save_path=os.path.join(PLOTS_DIR, "fprd_by_model_method.png")
)

## 26. Signed per-group FPRD heatmap

Unlike the Bias AUC heatmap (Section 14), FPRD has a meaningful zero (no disparity), so a
diverging colormap is the right choice here. Using PuOr (purple-orange) rather than a
red-green diverging map, for the same colorblind-accessibility reason as the rest of this
notebook. Covers all nine identity groups.

In [ ]:
def plot_signed_fprd_heatmap(fairness_df, save_path=None):
    if fairness_df.empty:
        print("No data to plot yet.")
        return

    pivot = fairness_df.pivot_table(
        index="identity_group",
        columns=["model_safe", "method"],
        values="fpr_diff",
    )
    pivot.columns = [f"{m}\n{meth}" for m, meth in pivot.columns]

    vmax = pivot.abs().max().max()
    fig, ax = plt.subplots(figsize=(max(10, len(pivot.columns) * 1.1), 6))
    sns.heatmap(
        pivot, annot=True, fmt=".3f", cmap="PuOr_r",
        vmin=-vmax, vmax=vmax, center=0, linewidths=0.5,
        ax=ax, cbar_kws={"label": "FPRD (+ = over-flagged; 0 is fairer)"}
    )
    ax.set_title("False Positive Rate Difference (FPRD) by group and method",
                 fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("")
    ax.set_ylabel("Identity group")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


plot_signed_fprd_heatmap(
    fairness_df,
    save_path=os.path.join(PLOTS_DIR, "fprd_heatmap_signed.png")
)

## 27. Mean per-group Subgroup, BPSN, and BNSP AUC (one panel per model)

Arithmetic means across identity groups, already present as `mean_subgroup_auc`,
`mean_bpsn_auc`, and `mean_bnsp_auc` in `master`. These average slightly above the overall
Bias AUC reported in Section 12, since that aggregate uses a p=-5 power mean instead of a
simple mean.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


def plot_auc_component_deltas(master, save_path=None):
    """
    Plot mean Subgroup AUC, BPSN AUC, and BNSP AUC
    as changes compared with each model's own baseline.

    Positive = better than baseline
    Negative = worse than baseline
    """

    if master.empty:
        print("Master dataframe is empty.")
        return

    required_cols = [
        "model_safe",
        "method",
        "mean_subgroup_auc",
        "mean_bpsn_auc",
        "mean_bnsp_auc",
    ]

    missing = [c for c in required_cols if c not in master.columns]
    if missing:
        print(f"Missing columns: {missing}")
        return

    df = master[required_cols].copy()

    METHOD_COLORS = {
        "adapter": "#56B4E9",
        "prompt_tuning": "#E69F00",
        "full_debiasing": "#CC79A7",
    }

    METHOD_LABELS = {
        "baseline": "Baseline",
        "adapter": "Adapter",
        "prompt_tuning": "Prompt tuning",
        "full_debiasing": "Full fine-tuning",
    }

    MODEL_LABELS = {
        "bert_base_uncased": "BERT",
        "distilbert_base_uncased": "DistilBERT",
        "roberta_base": "RoBERTa",
    }

    METRIC_LABELS = {
        "mean_subgroup_auc": "Subgroup AUC",
        "mean_bpsn_auc": "BPSN AUC",
        "mean_bnsp_auc": "BNSP AUC",
    }

    method_order = [
        "adapter",
        "prompt_tuning",
        "full_debiasing",
    ]

    model_order = [
        "bert_base_uncased",
        "distilbert_base_uncased",
        "roberta_base",
    ]

    metric_order = [
        "mean_subgroup_auc",
        "mean_bpsn_auc",
        "mean_bnsp_auc",
    ]

    # Keep only models and methods that exist in the dataframe
    model_order = [m for m in model_order if m in df["model_safe"].unique()]
    method_order = [m for m in method_order if m in df["method"].unique()]

    if not model_order:
        print("No known models found in master['model_safe'].")
        return

    if not method_order:
        print("No non-baseline methods found.")
        return

    # Convert to long format
    long_df = df.melt(
        id_vars=["model_safe", "method"],
        value_vars=metric_order,
        var_name="metric",
        value_name="auc"
    )

    # Get baseline value for each model and metric
    baseline_df = (
        long_df[long_df["method"] == "baseline"]
        [["model_safe", "metric", "auc"]]
        .rename(columns={"auc": "baseline_auc"})
    )

    if baseline_df.empty:
        print("No baseline rows found. Check that master['method'] contains 'baseline'.")
        return

    # Merge baseline values back
    plot_df = long_df.merge(
        baseline_df,
        on=["model_safe", "metric"],
        how="left"
    )

    # Remove rows without baseline
    plot_df = plot_df.dropna(subset=["baseline_auc"])

    # Difference from baseline
    plot_df["delta_auc"] = plot_df["auc"] - plot_df["baseline_auc"]

    # Convert to percentage points
    plot_df["delta_auc_points"] = plot_df["delta_auc"] * 100

    # Remove baseline from the plotted bars because baseline = 0
    plot_df = plot_df[plot_df["method"].isin(method_order)].copy()

    if plot_df.empty:
        print("No data left to plot after filtering methods.")
        return

    # Pretty labels
    plot_df["model_label"] = plot_df["model_safe"].map(MODEL_LABELS)
    plot_df["method_label"] = plot_df["method"].map(METHOD_LABELS)
    plot_df["metric_label"] = plot_df["metric"].map(METRIC_LABELS)

    sns.set_theme(style="whitegrid")

    g = sns.catplot(
        data=plot_df,
        x="metric_label",
        y="delta_auc_points",
        hue="method",
        col="model_label",
        kind="bar",
        order=[METRIC_LABELS[m] for m in metric_order],
        hue_order=method_order,
        col_order=[MODEL_LABELS[m] for m in model_order],
        palette=METHOD_COLORS,
        height=4.5,
        aspect=1.15,
        edgecolor="black",
        linewidth=0.7,
        legend=False
    )

    # Symmetric y-axis around zero
    max_abs = plot_df["delta_auc_points"].abs().max()
    ylim = max(0.5, np.ceil(max_abs * 1.4 * 10) / 10)

    for ax in g.axes.flat:
        ax.axhline(0, color="black", linewidth=1.2)
        ax.set_ylim(-ylim, ylim)

        ax.set_xlabel("")
        ax.set_ylabel("Δ AUC vs baseline\n(percentage points)", fontsize=10)

        ax.tick_params(axis="x", labelsize=9)
        ax.tick_params(axis="y", labelsize=9)

        # Add values on bars
        for container in ax.containers:
            labels = []

            for bar in container:
                value = bar.get_height()

                if np.isnan(value):
                    labels.append("")
                elif value > 0:
                    labels.append(f"+{value:.2f}")
                else:
                    labels.append(f"{value:.2f}")

            ax.bar_label(
                container,
                labels=labels,
                fontsize=8,
                padding=3
            )

    g.set_titles("{col_name}", size=11, weight="bold")

    # Custom legend with clean display names
    handles = [
        plt.Line2D(
            [0], [0],
            marker="s",
            color="w",
            markerfacecolor=METHOD_COLORS[m],
            markeredgecolor="black",
            markersize=10,
            label=METHOD_LABELS[m]
        )
        for m in method_order
    ]

    g.fig.legend(
        handles=handles,
        loc="lower center",
        ncol=len(method_order),
        frameon=True,
        bbox_to_anchor=(0.5, -0.03),
        fontsize=9
    )

    g.fig.suptitle(
        "Change in mean Bias AUC components compared with baseline",
        fontsize=15,
        fontweight="bold",
        y=1.06
    )

    g.fig.subplots_adjust(top=0.82, bottom=0.22)

    if save_path:
        g.fig.savefig(save_path, bbox_inches="tight", dpi=300)
        print(f"Saved: {save_path}")

    plt.show()


plot_auc_component_deltas(
    master,
    save_path=os.path.join(PLOTS_DIR, "auc_components_delta_vs_baseline.png")
)

## 28. Per-group EOD for the baseline condition, one panel per model

A filtered view of the same per-group EOD data used in Section 15 (all 12 runs), restricted
here to the baseline condition only, so the three backbones can be compared directly.

In [ ]:
def plot_eod_bars_single_method(fairness_df, method, save_path=None):
    sub_all = fairness_df[fairness_df.method == method]
    models = sub_all["model_safe"].unique()
    if len(models) == 0:
        print(f"No data for method='{method}' yet.")
        return

    fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 5), sharey=True)
    if len(models) == 1:
        axes = [axes]

    for ax, model_name in zip(axes, models):
        sub = sub_all[sub_all.model_safe == model_name].sort_values("eod")
        colors = ["#D55E00" if v > 0 else "#0072B2" for v in sub.eod]
        ax.barh(sub.identity_group, sub.eod, color=colors)
        ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
        ax.set_title(model_name, fontsize=11)
        ax.set_xlabel("EOD (+ = under-detection bias)")

    axes[0].set_ylabel("Identity group")
    fig.suptitle(f"Per-group Equality of Opportunity Difference ({method})",
                 fontweight="bold")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


plot_eod_bars_single_method(
    fairness_df, "baseline",
    save_path=os.path.join(PLOTS_DIR, "eod_bars_baseline_only.png")
)

## 29. Per-group EOD for the BERT baseline, with subgroup sample sizes

Same data as one panel of Section 28, for BERT specifically, with each bar annotated by its
identity-group sample size (`subgroup_size`), so extreme values from small groups (e.g.
psychiatric_or_mental_illness, jewish) can be flagged rather than taken at face value.

In [ ]:
def plot_eod_bars_annotated(fairness_df, model_safe_name, method, save_path=None):
    sub = fairness_df[
        (fairness_df.model_safe == model_safe_name) & (fairness_df.method == method)
    ].sort_values("eod")
    if sub.empty:
        print("No data for that model/method combination yet.")
        return

    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ["#D55E00" if v > 0 else "#0072B2" for v in sub.eod]
    bars = ax.barh(sub.identity_group, sub.eod, color=colors)
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")

    for bar, n in zip(bars, sub.subgroup_size):
        small_flag = " (small)" if n < MIN_GROUP_SIZE else ""
        x_pos = bar.get_width()
        offset = 0.01 if x_pos >= 0 else -0.01
        ha = "left" if x_pos >= 0 else "right"
        ax.text(x_pos + offset, bar.get_y() + bar.get_height() / 2,
                f"n={n}{small_flag}", va="center", ha=ha, fontsize=8)

    ax.set_xlabel("EOD (+ = under-detection bias)")
    ax.set_ylabel("Identity group")
    ax.set_title(f"Per-group EOD for the {model_safe_name} {method}, with subgroup sizes",
                 fontweight="bold")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


plot_eod_bars_annotated(
    fairness_df, "bert_base_uncased", "baseline",
    save_path=os.path.join(PLOTS_DIR, "eod_bars_bert_baseline_annotated.png")
)

## 30. Energy and trainable parameters, by model and method

Two grouped bar charts using columns already in `master`: energy consumption (linear scale)
and trainable parameter count (log scale, since it spans four orders of magnitude between
prompt tuning and full fine-tuning).

In [ ]:
def plot_energy_and_params(master, save_path_energy=None, save_path_params=None):
    if master.empty:
        print("No data to plot yet.")
        return

    METHOD_ORDER  = ["baseline", "adapter", "prompt_tuning", "full_debiasing"]
    METHOD_LABELS = {
        "baseline": "Baseline", "adapter": "Adapter",
        "prompt_tuning": "Prompt tuning", "full_debiasing": "Full fine-tuning",
    }
    METHOD_COLORS = {
        "baseline"      : "#7F7F7F",
        "adapter"       : "#56B4E9",
        "prompt_tuning" : "#E69F00",
        "full_debiasing": "#CC79A7",
    }

    def grouped_bar(value_col, ylabel, title, save_path, log_scale=False):
        if value_col not in master.columns or master[value_col].isna().all():
            print(f"No '{value_col}' data available yet.")
            return
        pivot = master.pivot_table(index="model_safe", columns="method", values=value_col)
        pivot = pivot.reindex(columns=[m for m in METHOD_ORDER if m in pivot.columns])

        fig, ax = plt.subplots(figsize=(8, 5))
        x = np.arange(len(pivot.index))
        width = 0.8 / len(pivot.columns)
        for i, method in enumerate(pivot.columns):
            ax.bar(x + i * width, pivot[method], width,
                   label=METHOD_LABELS.get(method, method),
                   color=METHOD_COLORS.get(method, "#999999"))
        ax.set_xticks(x + width * (len(pivot.columns) - 1) / 2)
        ax.set_xticklabels(pivot.index)
        if log_scale:
            ax.set_yscale("log")
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontweight="bold")
        ax.legend()
        plt.tight_layout()
        if save_path:
            fig.savefig(save_path, bbox_inches="tight")
            print(f"Saved: {save_path}")
        plt.show()

    grouped_bar("energy_kwh", "Energy (kWh)",
                "Energy consumption by model and method", save_path_energy)
    grouped_bar("trainable_parameters", "Trainable parameters (log scale)",
                "Trainable parameters by model and method", save_path_params, log_scale=True)


plot_energy_and_params(
    master,
    save_path_energy=os.path.join(PLOTS_DIR, "energy_by_model_method.png"),
    save_path_params=os.path.join(PLOTS_DIR, "trainable_params_by_model_method.png"),
)

## 31. Overall Bias AUC vs. energy consumption

Energy on the x-axis (not encoded as bubble size, unlike Section 16's plot). The point with
the lowest energy among configurations that beat their own model's baseline Bias AUC is
labelled automatically, rather than hardcoding which point that is, since it can change as
runs are repeated.

In [ ]:
def plot_bias_auc_vs_energy(master, save_path=None):
    if master.empty or "energy_kwh" not in master.columns:
        print("No energy data to plot yet.")
        return

    MODEL_COLORS = {
        "bert_base_uncased"       : "#0072B2",
        "distilbert_base_uncased" : "#E69F00",
        "roberta_base"            : "#009E73",
    }
    METHOD_MARKERS = {
        "baseline"      : "o",
        "prompt_tuning" : "s",
        "adapter"       : "^",
        "full_debiasing": "D",
    }

    fig, ax = plt.subplots(figsize=(8, 6))
    for _, row in master.iterrows():
        color  = MODEL_COLORS.get(row["model_safe"], "grey")
        marker = METHOD_MARKERS.get(row["method"], "x")
        ax.scatter(row["energy_kwh"], row["overall_bias_auc"],
                   color=color, marker=marker, s=120,
                   edgecolors="black", linewidths=0.6, zorder=3)

    # Auto-label the lowest-energy point among configs at or above their own model's baseline
    baselines = master[master.method == "baseline"].set_index("model_safe")["overall_bias_auc"]
    candidates = master[
        master.apply(lambda r: r["overall_bias_auc"] >= baselines.get(r["model_safe"], -1), axis=1)
    ]
    if not candidates.empty:
        best = candidates.loc[candidates["energy_kwh"].idxmin()]
        ax.annotate(
            f"{best['model_safe']} + {best['method']}\n(lowest energy,\nabove-baseline fairness)",
            (best["energy_kwh"], best["overall_bias_auc"]),
            textcoords="offset points", xytext=(15, 10), fontsize=8,
            arrowprops=dict(arrowstyle="->", lw=0.8)
        )

    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=10,
               label=m.replace("_", " ")) for m, c in MODEL_COLORS.items()
    ] + [
        Line2D([0], [0], marker=mk, color="grey", markersize=9, label=mt.replace("_", " "))
        for mt, mk in METHOD_MARKERS.items()
    ]
    ax.legend(handles=legend_elements, fontsize=8, loc="best")

    ax.set_xlabel("Energy (kWh) (lower is greener)", fontsize=11)
    ax.set_ylabel("Overall Bias AUC (higher is fairer)", fontsize=11)
    ax.set_title("Overall Bias AUC vs. energy consumption, by model and method",
                 fontweight="bold")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


plot_bias_auc_vs_energy(
    master,
    save_path=os.path.join(PLOTS_DIR, "bias_auc_vs_energy.png")
)

## 32. Precision vs. recall, by model and method

F1 alone hides the operating-point shift discussed in the Performance section: adapters and
prompt tuning trade precision for recall rather than simply losing classification ability.
This makes that shift visible directly, on the same visual language as the trade-off plots.

In [ ]:
def plot_precision_vs_recall(master, save_path=None):
    if master.empty:
        print("No data to plot yet.")
        return

    MODEL_COLORS = {
        "bert_base_uncased"       : "#0072B2",
        "distilbert_base_uncased" : "#E69F00",
        "roberta_base"            : "#009E73",
    }
    METHOD_MARKERS = {
        "baseline"       : "o",
        "prompt_tuning"  : "s",
        "adapter"        : "^",
        "full_debiasing": "D",
    }

    fig, ax = plt.subplots(figsize=(8, 6))
    for _, row in master.iterrows():
        color  = MODEL_COLORS.get(row["model_safe"], "grey")
        marker = METHOD_MARKERS.get(row["method"], "x")
        ax.scatter(row["recall"], row["precision"],
                   color=color, marker=marker, s=120,
                   edgecolors="black", linewidths=0.6, zorder=3)

    # Diagonal reference: equal precision and recall
    lims = [
        min(master["recall"].min(), master["precision"].min()) - 0.02,
        max(master["recall"].max(), master["precision"].max()) + 0.02,
    ]
    ax.plot(lims, lims, color="grey", linestyle="--", linewidth=0.8, alpha=0.6, zorder=1)

    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=10,
               label=m.replace("_", " ")) for m, c in MODEL_COLORS.items()
    ] + [
        Line2D([0], [0], marker=mk, color="grey", markersize=9, label=mt.replace("_", " "))
        for mt, mk in METHOD_MARKERS.items()
    ]
    ax.legend(handles=legend_elements, fontsize=8, loc="best")

    ax.set_xlabel("Recall", fontsize=11)
    ax.set_ylabel("Precision", fontsize=11)
    ax.set_title("Precision vs. recall, by model and method", fontweight="bold")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


plot_precision_vs_recall(
    master,
    save_path=os.path.join(PLOTS_DIR, "precision_vs_recall.png")
)

## 33. Overall Bias AUC by model and method

The headline fairness aggregate cited throughout the Discussion (the p=-5 power mean of
Subgroup, BPSN, and BNSP AUC), shown on its own rather than only inside the combined
trade-off scatter.

In [ ]:
def plot_overall_bias_auc(master, save_path=None):
    if master.empty:
        print("No data to plot yet.")
        return

    METHOD_ORDER  = ["baseline", "adapter", "prompt_tuning", "full_debiasing"]
    METHOD_LABELS = {
        "baseline": "Baseline", "adapter": "Adapter",
        "prompt_tuning": "Prompt tuning", "full_debiasing": "Full fine-tuning",
    }
    METHOD_COLORS = {
        "baseline"       : "#7F7F7F",
        "adapter"        : "#56B4E9",
        "prompt_tuning"  : "#E69F00",
        "full_debiasing": "#CC79A7",
    }

    pivot = master.pivot_table(index="model_safe", columns="method", values="overall_bias_auc")
    pivot = pivot.reindex(columns=[m for m in METHOD_ORDER if m in pivot.columns])

    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(pivot.index))
    width = 0.8 / len(pivot.columns)
    for i, method in enumerate(pivot.columns):
        ax.bar(x + i * width, pivot[method], width,
               label=METHOD_LABELS.get(method, method),
               color=METHOD_COLORS.get(method, "#999999"))
    ax.set_xticks(x + width * (len(pivot.columns) - 1) / 2)
    ax.set_xticklabels(pivot.index)
    ax.set_ylim(pivot.values.min() - 0.02, pivot.values.max() + 0.01)
    ax.set_ylabel("Overall Bias AUC (higher is fairer)")
    ax.set_title("Overall Bias AUC by model and method", fontweight="bold")
    ax.legend()
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


plot_overall_bias_auc(
    master,
    save_path=os.path.join(PLOTS_DIR, "overall_bias_auc_by_model_method.png")
)

## 34. Pareto frontier across performance, fairness, and sustainability

For the Discussion section that combines all three axes. A configuration is on the frontier
if no other configuration is at least as good on F1, Bias AUC, *and* energy, and strictly
better on at least one. Same axes as the F1-vs-Bias-AUC trade-off plot, with the non-dominated set highlighted instead of drawn as a separate figure, so the two are
directly comparable.

In [ ]:
def compute_pareto_frontier(master):
    """A row is on the frontier if no other row beats-or-ties it on f1 and bias_auc
    while beating-or-tying it on (lower) energy, with at least one strict improvement."""
    rows = master.to_dict("records")
    frontier_flags = []
    for i, a in enumerate(rows):
        dominated = False
        for j, b in enumerate(rows):
            if i == j:
                continue
            not_worse = (
                b["f1_binary"] >= a["f1_binary"] and
                b["overall_bias_auc"] >= a["overall_bias_auc"] and
                b["energy_kwh"] <= a["energy_kwh"]
            )
            strictly_better = (
                b["f1_binary"] > a["f1_binary"] or
                b["overall_bias_auc"] > a["overall_bias_auc"] or
                b["energy_kwh"] < a["energy_kwh"]
            )
            if not_worse and strictly_better:
                dominated = True
                break
        frontier_flags.append(not dominated)
    return frontier_flags


def plot_pareto_frontier(master, save_path=None):
    if master.empty or "energy_kwh" not in master.columns:
        print("No energy data to plot yet.")
        return

    MODEL_COLORS = {
        "bert_base_uncased"       : "#0072B2",
        "distilbert_base_uncased" : "#E69F00",
        "roberta_base"            : "#009E73",
    }
    METHOD_MARKERS = {
        "baseline"       : "o",
        "prompt_tuning"  : "s",
        "adapter"        : "^",
        "full_debiasing": "D",
    }

    on_frontier = compute_pareto_frontier(master)
    fig, ax = plt.subplots(figsize=(9, 7))

    for (_, row), frontier in zip(master.iterrows(), on_frontier):
        color  = MODEL_COLORS.get(row["model_safe"], "grey")
        marker = METHOD_MARKERS.get(row["method"], "x")
        if frontier:
            ax.scatter(row["f1_binary"], row["overall_bias_auc"],
                       color=color, marker=marker, s=220,
                       edgecolors="black", linewidths=1.8, zorder=4)
            ax.annotate(f"{row['model_safe'][:4]}/{row['method']}",
                        (row["f1_binary"], row["overall_bias_auc"]),
                        textcoords="offset points", xytext=(7, 5), fontsize=7, fontweight="bold")
        else:
            ax.scatter(row["f1_binary"], row["overall_bias_auc"],
                       color=color, marker=marker, s=90, alpha=0.35,
                       edgecolors="grey", linewidths=0.5, zorder=2)

    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=10,
               label=m.replace("_", " ")) for m, c in MODEL_COLORS.items()
    ] + [
        Line2D([0], [0], marker=mk, color="grey", markersize=9, label=mt.replace("_", " "))
        for mt, mk in METHOD_MARKERS.items()
    ] + [
        Line2D([0], [0], marker="o", color="w", markerfacecolor="grey",
               markeredgecolor="black", markeredgewidth=1.8, markersize=12,
               label="On the frontier"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="grey", alpha=0.35,
               markersize=9, label="Dominated"),
    ]
    ax.legend(handles=legend_elements, fontsize=8, loc="best")

    ax.set_xlabel("F1 Binary (higher is better)", fontsize=11)
    ax.set_ylabel("Overall Bias AUC (higher is fairer)", fontsize=11)
    ax.set_title(
        "Pareto frontier: F1, Bias AUC, and energy jointly\n"
        "(bold outline = not beaten on all three axes at once)",
        fontweight="bold"
    )
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()

    n_frontier = sum(on_frontier)
    print(f"{n_frontier} of {len(master)} configurations are on the frontier "
          f"(not dominated on F1, Bias AUC, and energy jointly).")


plot_pareto_frontier(
    master,
    save_path=os.path.join(PLOTS_DIR, "pareto_frontier.png")
)

## 35. Cross-axis rank comparison

The most literal version of the Discussion's "no method is jointly optimal" claim: for each
method, its average rank (1 = best) on F1, Bias AUC, mean absolute FPRD, and energy,
averaged across the three backbones. No single row should read all 1s.

In [ ]:
# ── Presentation/backup plot: Subgroup AUC heatmap across all models ───────────

def plot_subgroup_heatmap_all_models(fairness_df):
    df = fairness_df.copy()

    # Normalize old method names
    df["method"] = df["method"].replace({
        "full_debiasing": "full_finetuning",
        "cda_full_finetuning": "full_finetuning",
        "full_ft": "full_finetuning"
    })

    # Model display names
    model_labels = {
        "bert_base_uncased": "BERT",
        "distilbert_base_uncased": "DistilBERT",
        "roberta_base": "RoBERTa"
    }

    # Method display names
    method_labels = {
        "baseline": "Baseline",
        "adapter": "Adapter",
        "prompt_tuning": "Prompt tuning",
        "full_finetuning": "Full finetuning"
    }

    df["model_label"] = df["model_safe"].map(model_labels)
    df["method_label"] = df["method"].map(method_labels)

    # Drop rows that could not be mapped
    df = df.dropna(subset=["model_label", "method_label"])

    model_order = ["BERT", "DistilBERT", "RoBERTa"]
    method_order = ["Baseline", "Adapter", "Prompt tuning", "Full finetuning"]

    # Create combined column label: Model + Method
    df["model_method"] = df["model_label"] + " | " + df["method_label"]

    column_order = [
        f"{model} | {method}"
        for model in model_order
        for method in method_order
    ]

    # Keep only existing columns
    column_order = [c for c in column_order if c in df["model_method"].unique()]

    pivot = df.pivot_table(
        index="identity_group",
        columns="model_method",
        values="subgroup_auc",
        aggfunc="mean"
    )

    pivot = pivot[column_order]

    # Optional: nicer row order
    identity_order = [
        "male",
        "female",
        "homosexual_gay_or_lesbian",
        "christian",
        "jewish",
        "muslim",
        "black",
        "white",
        "psychiatric_or_mental_illness"
    ]

    pivot = pivot.reindex([g for g in identity_order if g in pivot.index])

    plt.figure(figsize=(16, 6))

    ax = sns.heatmap(
        pivot,
        annot=True,
        fmt=".3f",
        cmap="viridis",
        vmin=0.75,
        vmax=0.95,
        linewidths=0.5,
        cbar_kws={"label": "Subgroup AUC (higher is fairer)"}
    )

    # Show only method names on x-axis, repeated per model
    short_xticklabels = [col.split(" | ")[1] for col in pivot.columns]
    ax.set_xticklabels(short_xticklabels, rotation=45, ha="right", fontsize=8)

    ax.set_title(
        "Subgroup AUC by identity group and method",
        fontsize=14,
        fontweight="bold",
        pad=18
    )

    ax.set_xlabel("")
    ax.set_ylabel("")

    plt.tight_layout()

    save_path = os.path.join(PLOTS_DIR, "subgroup_auc_all_models_heatmap.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {save_path}")

    plt.show()


plot_subgroup_heatmap_all_models(fairness_df)

In [ ]:
# ── Presentation/backup plot: FPRD heatmap across all models ───────────────────

def plot_fprd_heatmap_all_models(fairness_df):
    df = fairness_df.copy()

    # Normalize old method names
    df["method"] = df["method"].replace({
        "full_debiasing": "full_finetuning",
        "cda_full_finetuning": "full_finetuning",
        "full_ft": "full_finetuning"
    })

    # Model display names
    model_labels = {
        "bert_base_uncased": "BERT",
        "distilbert_base_uncased": "DistilBERT",
        "roberta_base": "RoBERTa"
    }

    # Method display names
    method_labels = {
        "baseline": "Baseline",
        "adapter": "Adapter",
        "prompt_tuning": "Prompt tuning",
        "full_finetuning": "Full finetuning"
    }

    df["model_label"] = df["model_safe"].map(model_labels)
    df["method_label"] = df["method"].map(method_labels)

    # Drop rows that could not be mapped
    df = df.dropna(subset=["model_label", "method_label"])

    model_order = ["BERT", "DistilBERT", "RoBERTa"]
    method_order = ["Baseline", "Adapter", "Prompt tuning", "Full finetuning"]

    # Pick the correct FPRD column automatically
    possible_fprd_cols = [
        "fprd",
        "fpr_difference",
        "fpr_diff",
        "fpr_parity_diff",
        "false_positive_rate_difference"
    ]

    metric_col = None
    for col in possible_fprd_cols:
        if col in df.columns:
            metric_col = col
            break

    if metric_col is None:
        raise ValueError(
            f"No FPRD column found. Available columns are:\n{list(df.columns)}"
        )

    # Create combined column label: Model + Method
    df["model_method"] = df["model_label"] + " | " + df["method_label"]

    column_order = [
        f"{model} | {method}"
        for model in model_order
        for method in method_order
    ]

    # Keep only existing columns
    column_order = [c for c in column_order if c in df["model_method"].unique()]

    pivot = df.pivot_table(
        index="identity_group",
        columns="model_method",
        values=metric_col,
        aggfunc="mean"
    )

    pivot = pivot[column_order]

    # Optional: row order matching your thesis plots
    identity_order = [
        "male",
        "female",
        "homosexual_gay_or_lesbian",
        "christian",
        "jewish",
        "muslim",
        "black",
        "white",
        "psychiatric_or_mental_illness"
    ]

    pivot = pivot.reindex([g for g in identity_order if g in pivot.index])

    plt.figure(figsize=(16, 6))

    ax = sns.heatmap(
        pivot,
        annot=True,
        fmt=".3f",
        cmap="magma",
        vmin=0.00,
        vmax=0.45,
        linewidths=0.5,
        cbar_kws={"label": "FPRD lower is fairer"}
    )

    # Show only method names on x-axis, repeated for each model
    short_xticklabels = [col.split(" | ")[1] for col in pivot.columns]
    ax.set_xticklabels(short_xticklabels, rotation=45, ha="right", fontsize=8)

    ax.set_title(
        "False Positive Rate Difference (FPRD) by group and method",
        fontsize=14,
        fontweight="bold",
        pad=18
    )

    ax.set_xlabel("")
    ax.set_ylabel("")

    plt.tight_layout()

    save_path = os.path.join(PLOTS_DIR, "fprd_all_models_heatmap.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {save_path}")

    plt.show()


plot_fprd_heatmap_all_models(fairness_df)

In [ ]:
def plot_rank_comparison(master, fairness_df, save_path=None):
    if master.empty:
        print("No data to plot yet.")
        return

    METHOD_ORDER  = ["baseline", "adapter", "prompt_tuning", "full_debiasing"]
    METHOD_LABELS = {
        "baseline": "Baseline", "adapter": "Adapter",
        "prompt_tuning": "Prompt tuning", "full_debiasing": "Full fine-tuning",
    }

    mean_abs_fprd = (
        fairness_df
        .groupby(["model_safe", "method"])["fpr_diff"]
        .apply(lambda s: s.abs().mean())
        .reset_index(name="mean_abs_fprd")
    )
    df = master.merge(mean_abs_fprd, on=["model_safe", "method"], how="left")

    # Rank within each backbone: higher-is-better metrics ranked descending,
    # lower-is-better metrics (FPRD, energy) ranked ascending. 1 = best in every case.
    df["rank_f1"]       = df.groupby("model_safe")["f1_binary"].rank(ascending=False)
    df["rank_bias_auc"] = df.groupby("model_safe")["overall_bias_auc"].rank(ascending=False)
    df["rank_fprd"]     = df.groupby("model_safe")["mean_abs_fprd"].rank(ascending=True)
    df["rank_energy"]   = df.groupby("model_safe")["energy_kwh"].rank(ascending=True)

    rank_cols   = ["rank_f1", "rank_bias_auc", "rank_fprd", "rank_energy"]
    col_labels  = ["F1", "Bias AUC", "Mean\nabs. FPRD", "Energy"]
    avg_ranks   = df.groupby("method")[rank_cols].mean()
    avg_ranks   = avg_ranks.reindex([m for m in METHOD_ORDER if m in avg_ranks.index])
    avg_ranks.index = [METHOD_LABELS.get(m, m) for m in avg_ranks.index]

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(
        avg_ranks, annot=True, fmt=".2f", cmap="viridis_r",
        vmin=1, vmax=4, linewidths=0.5, ax=ax,
        cbar_kws={"label": "Average rank (1 = best, 4 = worst)"},
        xticklabels=col_labels,
    )
    ax.set_title("Average rank across axes, by method\n(no row should read all 1s)",
                 fontsize=12, fontweight="bold", pad=12)
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


plot_rank_comparison(
    master, fairness_df,
    save_path=os.path.join(PLOTS_DIR, "cross_axis_rank_comparison.png")
)

## 36. Mean mutual information by model and method

MI(identity-term presence, predicted label), averaged across identity groups, per
model and method. Cited extensively in the Discussion but previously had no
supporting figure.

In [ ]:
def plot_mi_by_model_method(mi_csv_path, save_path=None):
    mi_df = pd.read_csv(mi_csv_path)
    if mi_df.empty:
        print("No mutual information data to plot yet.")
        return

    METHOD_ORDER  = ["baseline", "adapter", "prompt_tuning", "full_debiasing"]
    METHOD_LABELS = {
        "baseline": "Baseline", "adapter": "Adapter",
        "prompt_tuning": "Prompt tuning", "full_debiasing": "Full fine-tuning",
    }
    METHOD_COLORS = {
        "baseline"      : "#7F7F7F",
        "adapter"       : "#56B4E9",
        "prompt_tuning" : "#E69F00",
        "full_debiasing": "#CC79A7",
    }

    mean_mi = (
        mi_df.groupby(["model_safe", "method"])["mutual_information"]
        .mean()
        .reset_index(name="mean_mi")
    )
    pivot = mean_mi.pivot_table(index="model_safe", columns="method", values="mean_mi")
    pivot = pivot.reindex(columns=[m for m in METHOD_ORDER if m in pivot.columns])

    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(pivot.index))
    width = 0.8 / len(pivot.columns)
    for i, method in enumerate(pivot.columns):
        ax.bar(x + i * width, pivot[method], width,
               label=METHOD_LABELS.get(method, method),
               color=METHOD_COLORS.get(method, "#999999"))
    ax.set_xticks(x + width * (len(pivot.columns) - 1) / 2)
    ax.set_xticklabels(pivot.index)
    ax.set_ylabel("Mean mutual information")
    ax.set_title("Mean mutual information (identity term vs. predicted label)\nby model and method", fontweight="bold")
    ax.legend()
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


plot_mi_by_model_method(
    "/Users/medsa/Desktop/finaline/mutual_information.csv",
    save_path="/Users/medsa/Desktop/finaline/mi_by_model_method.png"
)